In [15]:
from pathlib import Path

import boto3
import pandas as pd
import numpy as np
from botocore.exceptions import BotoCoreError, ClientError

from secretkeys import ACCESS_KEY, KEY_ID

# Замените примеры значениями преподавателя.
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/listing.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

In [16]:
def read_s3_csv(bucket, key):

    response=s3.get_object(Bucket=bucket, Key=key)
    df=pd.read_csv(response['Body'])
    return df

In [17]:
def read_data(bucket, key, local_path=None):
    try:
        df=read_s3_csv(bucket, key)
        return df

    except (BotoCoreError, ClientError, OSError) as e:
        if local_path is None:
            raise

        print(f"S3 error: {type(e).__name__}: {e}")
        print(f"Reading local file: {local_path}")

        df=pd.read_csv(local_path)
        return df

In [18]:
df_s3 = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,1001248165793428398,2026-03-21,51.434110,-0.161020,2,1.0,1.0,1.0,1.0,Wandsworth,Entire serviced apartment,Entire home/apt,130.0
1,1001501494788436614,2026-03-21,51.514120,-0.176810,5,1.0,3.0,3.0,2.0,Westminster,Entire rental unit,Entire home/apt,232.0
2,1001555419418023705,2026-03-21,51.542042,0.121361,4,2.0,2.0,2.0,1.0,Barking and Dagenham,Entire condo,Entire home/apt,216.0
3,1001555573812356101,2026-03-21,51.540557,0.120994,2,1.0,1.0,1.0,1.0,Barking and Dagenham,Entire place,Entire home/apt,115.0
4,1001604905732387546,2026-03-21,51.377860,-0.086780,2,1.0,1.0,1.0,1.0,Croydon,Private room in home,Private room,86.0


Размер таблицы: (5724, 13)


In [19]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

In [20]:
X=df_s3.drop(['listing_id', 'snapshot_date', 'price'], axis=1)
y=df_s3['price']

In [21]:
X_train, X_test, y_train, y_test=train_test_split(X, y, random_state=42, test_size=0.25, shuffle=True)

In [22]:
num_cols=X_train.select_dtypes(include=('number')).columns
cat_cols=X_train.select_dtypes(exclude=('number')).columns

In [23]:
num_pipeline=Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline,num_cols),
        ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ],
    remainder='passthrough'
)

In [25]:
pipe=Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression(fit_intercept=True, n_jobs=-1))
])

In [26]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

c:\Users\zhiga\anaconda3\envs\mlops26\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [27]:
RMSE=root_mean_squared_error(y_test, y_pred)
print('RMSE: ', np.round(RMSE, 4))

RMSE:  267.9366


In [28]:
import joblib

joblib.dump(pipe, "models/model.pkl")

['models/model.pkl']